# DOA Pipeline Demo

Streams `data/static_10m_000.wav` through two parallel DOA chains and prints every chunk.

**Outer ring (CH1–4):** raw audio → STFT → DOA `freq_range=[50, 357]` Hz  
*(no HP filter — HP would remove the only unambiguous zone below the 357 Hz aliasing limit)*

**Inner ring (CH5–8):** HP 400 Hz → STFT → DOA `freq_range=[400, 808]` Hz  
*(HP removes wind; 400–808 Hz is clean and unambiguous below the 808 Hz aliasing limit)*

**File:** 8 channels, 44100 Hz, ~1.08 s

In [1]:
import sys
sys.path.insert(0, '..')  # make logic/ importable from notebooks/

import numpy as np
from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor, StftChunk
from logic.doa_processor import DoaProcessor
from logic.filters import HighPassFilter


In [ ]:
# --- Config ---
WAV_FILE    = '../data/static_10m_090.wav'
SAMPLE_RATE = 44100
CHUNK_SIZE  = 8192
NPERSEG     = 512
NOVERLAP    = 256

OUTER_IDX = slice(0, 4)   # CH1-4
INNER_IDX = slice(4, 8)   # CH5-8

# Outer ring (CH1-4): 34 cm square
# Spatial aliasing limit: c/(2*d_max) = 343/(2*0.481) ≈ 357 Hz
L_outer = np.array([
    [-0.17,  0.17, -0.17,  0.17],
    [-0.17, -0.17,  0.17,  0.17],
    [ 0.00,  0.00,  0.00,  0.00],
])
OUTER_FREQ_RANGE = [50, 357]   # Hz — stay below aliasing limit

# Inner ring (CH5-8): 15 cm square
# Spatial aliasing limit: c/(2*d_max) = 343/(2*0.212) ≈ 808 Hz
L_inner = np.array([
    [-0.075,  0.075, -0.075,  0.075],
    [-0.075, -0.075,  0.075,  0.075],
    [ 0.000,  0.000,  0.000,  0.000],
])
HP_CUTOFF_HZ     = 400             # removes wind below 400 Hz
INNER_FREQ_RANGE = [400, 808]      # Hz — above wind, below aliasing limit


In [3]:
# --- Processors ---
wav_outer  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
wav_inner  = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)

stft_outer = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
stft_inner = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)

hp_filter  = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)

doa_outer  = DoaProcessor(mic_locs=L_outer, sampling_rate=SAMPLE_RATE, nfft=NPERSEG,
                           freq_range=OUTER_FREQ_RANGE)
doa_inner  = DoaProcessor(mic_locs=L_inner, sampling_rate=SAMPLE_RATE, nfft=NPERSEG,
                           freq_range=INNER_FREQ_RANGE)


In [4]:
SEP = '─' * 76

def outer_stft_stream():
    """Raw audio → STFT (outer ring channels)."""
    for chunk in stft_outer.process(wav_outer.stream()):
        print(f"{chunk.timestamp:7.3f}s  AudioChunk → StftChunk  "
              f"shape={chunk.magnitudes.shape}  "
              f"freqs={chunk.freqs[0]:.0f}–{chunk.freqs[-1]:.0f} Hz")
        yield StftChunk(freqs=chunk.freqs, times=chunk.times,
                        magnitudes=chunk.magnitudes[OUTER_IDX],
                        sampling_rate=chunk.sampling_rate, timestamp=chunk.timestamp)

def inner_stft_stream():
    """HP-filtered audio → STFT (inner ring channels)."""
    for chunk in stft_inner.process(hp_filter.process(wav_inner.stream())):
        yield StftChunk(freqs=chunk.freqs, times=chunk.times,
                        magnitudes=chunk.magnitudes[INNER_IDX],
                        sampling_rate=chunk.sampling_rate, timestamp=chunk.timestamp)

# Build inner DOA dict keyed by timestamp for side-by-side comparison
inner_doa = {d.timestamp: d.azimuth_deg
             for d in doa_inner.process(inner_stft_stream())}

print(SEP)
print(f"{'timestamp':<10}  {'outer DOA':>12}  {'inner DOA':>12}  note")
print(SEP)

for doa_chunk in doa_outer.process(outer_stft_stream()):
    t  = doa_chunk.timestamp
    o  = doa_chunk.azimuth_deg
    i  = inner_doa.get(t, float('nan'))
    note = '← inner HP+filtered, freq-limited' if not np.isnan(i) else ''
    print(f"{t:7.3f}s   outer={o:6.1f}°   inner={i:6.1f}°  {note}")

print(SEP)


────────────────────────────────────────────────────────────────────────────
timestamp      outer DOA     inner DOA  note
────────────────────────────────────────────────────────────────────────────
  0.000s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.000s   outer=  82.0°   inner= 358.0°  ← inner HP+filtered, freq-limited
  0.186s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.186s   outer= 126.0°   inner= 358.0°  ← inner HP+filtered, freq-limited
  0.372s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.372s   outer= 131.0°   inner= 344.0°  ← inner HP+filtered, freq-limited
  0.557s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.557s   outer= 329.0°   inner= 359.0°  ← inner HP+filtered, freq-limited
  0.743s  AudioChunk → StftChunk  shape=(8, 257, 33)  freqs=0–22050 Hz
  0.743s   outer= 288.0°   inner=  15.0°  ← inner HP+filtered, freq-limited
  0.929s  AudioChunk → StftChunk  shape=(8, 257, 27)  freqs=0–2205